# Experiment 2: position-resolved feature ablation on matched bigram sentences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreyaph16/Mechanistic-Interpretability-Research-/blob/main/Experiment_2_Position_Resolved_Bigrams.ipynb)

**Model:** Gemma-2B · **SAEs:** `gemma-2b-res-jb` at the residual stream after layers 6 and 12

**Question.** In Experiment 1, almost every strong link between a layer-6 SAE feature and a layer-12 SAE feature sat at the *same* token, which fits the residual stream mostly carrying each token forward. Two weaker links crossed tokens. Are cross-position links like those real, and are they stronger when the first word predicts the second (*peanut → butter*) than when the two words are merely next to each other (*cold → butter*)?

**What changed from Experiment 1**

| Experiment 1 (pilot) | Experiment 2 |
|---|---|
| Effect = change in a feature's max over the whole sentence, including position 0 | Effect measured **at every position separately**; position 0 (BOS) excluded everywhere |
| A feature was ablated at every position at once | Each feature is ablated **at one position only**, so each link has a known source and target token |
| Token positions inferred afterwards from where features peak | Positions come from the measurement itself |
| One sentence | **18 matched pairs (36 sentences)**: the same frame with a predictable vs a merely adjacent first word |
| No baseline | **Random-direction control**: the same ablation with a random vector of the same size |
| — | **Next-token check**: does removing the feature change what the model predicts? |

## Design

For every sentence:

1. Run the model and encode the layer-6 and layer-12 residual streams with their SAEs.
2. Pick candidate features **per position** (positions ≥ 1): the top `K_PER_POS` layer-6 features at each token are the ones we ablate; the top `K_PER_POS` layer-12 features at each token are the ones we measure.
3. For each candidate (layer-6 feature *f*, position *p*): subtract *f*'s contribution from the layer-6 residual stream **at position p only**, rerun to layer 12, and record how much each measured layer-12 feature *g* drops at every position *q*.
4. **Effect size** = drop ÷ that layer-12 feature's peak activation in the sentence (positions ≥ 1). An effect of 0.10 means the feature lost 10% of its peak.
5. **Control:** repeat step 3 with a random direction of the same length as *f*'s contribution.

A link (f, p) → (g, q) is **same-position** if q = p and **cross-position** if q > p. Because Gemma is causal, q < p must show no effect at all; the notebook checks this as a sanity test of the ablation code.

**Predictions, written before running.** If the cross-position links in Experiment 1 reflect real computation tied to word predictability, then:

- (A) cross-position links should be clearly more common for real features than for random directions;
- (B) the word 1 → word 2 link should be stronger for predictable pairs than for their matched controls;
- (C) ablating that word-1 feature should lower the model's probability of word 2 more for predictable pairs.

If none of these hold, the Experiment 1 cross-position links were most likely noise or a side effect of how positions were inferred.

**Runtime.** Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*): a few minutes. On CPU it works but takes much longer.

## Setup

In [ ]:
# TransformerLens 4 removed HookedTransformer, so pin the 3.x line
!pip install -q "sae-lens>=6,<7" "transformer-lens>=3,<4" scipy

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display

torch.set_grad_enabled(False)
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EARLY_LAYER, LATE_LAYER = 6, 12
EARLY_HOOK = f"blocks.{EARLY_LAYER}.hook_resid_post"
LATE_HOOK = f"blocks.{LATE_LAYER}.hook_resid_post"

K_PER_POS = 5                       # candidate features per position, per layer
THRESHOLDS = [0.05, 0.10, 0.25]     # a link is "strong" if the late feature loses at least this share of its peak
MAIN_T = 0.10                       # threshold used for the headline numbers and plots
BATCH = 16                          # ablations run in parallel; lower this if you run out of memory

FEATURE_COLOR, RANDOM_COLOR = "#2a78d6", "#8a8a85"
PRED_COLOR, CTRL_COLOR = "#2a78d6", "#eb6834"

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


device: cpu


In [ ]:
from huggingface_hub import login
login()  # paste a Hugging Face token with access to google/gemma-2b when prompted

In [ ]:
from transformer_lens import HookedTransformer
from sae_lens import SAE

model = HookedTransformer.from_pretrained("gemma-2b", device=device)

def load_sae(hook_name):
    out = SAE.from_pretrained(release="gemma-2b-res-jb", sae_id=hook_name, device=device)
    return out[0] if isinstance(out, tuple) else out   # older sae_lens versions return (sae, cfg, sparsity)

sae_early = load_sae(EARLY_HOOK)
sae_late = load_sae(LATE_HOOK)

/tmp/ipykernel_1422/306625664.py:4: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gemma-2b", device=device)


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

... (12 outputs left out)

gemma_2b_blocks.12.hook_resid_post_16384(…): reconstructing file:   0%|          |  0.00B /  269MB            

gemma_2b_blocks.12.hook_resid_post_16384(…): downloading bytes:           |  0.00B            

gemma_2b_blocks.12.hook_resid_post_16384(…): reconstructing file:   0%|          |  0.00B / 65.6kB            

gemma_2b_blocks.12.hook_resid_post_16384(…): downloading bytes:           |  0.00B            

## Stimuli: matched pairs

Each frame appears twice, differing **only in word 1**. Word 2 and everything around it stay identical, so any difference in the word 1 → word 2 link comes from word 1.

Pairs are kept only if both versions tokenise word 1 and word 2 as single tokens at consecutive positions; anything dropped is printed below.

In [ ]:
PAIRS = [
    # frame ({} = word 1)                                predictable  control    word 2
    ("She spread the {} butter on her toast.",            "peanut",    "cold",    "butter"),
    ("We bought some {} cream after dinner.",             "ice",       "thick",   "cream"),
    ("He paid with his {} card at the store.",            "credit",    "plastic", "card"),
    ("The car was stuck in a {} jam for hours.",          "traffic",   "long",    "jam"),
    ("The kids played in the {} pool all afternoon.",     "swimming",  "large",   "pool"),
    ("They planned a {} party for Friday.",               "birthday",  "small",   "party"),
    ("We met at the {} shop near the station.",           "coffee",    "new",     "shop"),
    ("She teaches at the {} school in town.",             "high",      "local",   "school"),
    ("They sat in the {} room after lunch.",              "living",    "quiet",   "room"),
    ("He left the car in the {} lot overnight.",          "parking",   "empty",   "lot"),
    ("She drank a glass of {} juice this morning.",       "orange",    "fresh",   "juice"),
    ("He fastened his {} belt before driving.",           "seat",      "leather", "belt"),
    ("He bought {} paper at the shop.",                   "toilet",    "white",   "paper"),
    ("She mailed the letter at the {} office yesterday.", "post",      "main",    "office"),
    ("They ate dinner at the {} table together.",         "dining",    "wooden",  "table"),
    ("His {} clock rang at six.",                         "alarm",     "old",     "clock"),
    ("A {} bear walked across the ice.",                  "polar",     "huge",    "bear"),
    ("A {} truck drove past the school.",                 "fire",      "red",     "truck"),
]

def to_ids(text):
    return model.to_tokens(text)[0].tolist()   # includes BOS at position 0

def locate_pair(frame, w1, w2):
    """Tokens plus positions of word 1 and word 2, or None if either is not a single token."""
    prefix = frame.split("{}")[0].rstrip()
    A = to_ids(prefix)
    B = to_ids(f"{prefix} {w1}")
    C = to_ids(f"{prefix} {w1} {w2}")
    S = to_ids(frame.format(w1))
    ok = (len(B) == len(A) + 1 and B[:len(A)] == A
          and len(C) == len(B) + 1 and C[:len(B)] == B
          and S[:len(C)] == C and len(S) > len(C))   # a token must follow word 2
    return (torch.tensor([S], device=device), len(A), len(B)) if ok else None

stimuli, dropped = [], []
for pair_id, (frame, w1_pred, w1_ctrl, w2) in enumerate(PAIRS):
    assert "{} " + w2 in frame, frame
    versions = {"predictable": w1_pred, "control": w1_ctrl}
    found = {cond: locate_pair(frame, w1, w2) for cond, w1 in versions.items()}
    if any(v is None for v in found.values()):
        dropped.append(frame.format(f"[{w1_pred}/{w1_ctrl}]"))
        continue
    for cond, w1 in versions.items():
        tokens, p1, p2 = found[cond]
        stimuli.append(dict(pair_id=pair_id, condition=cond, sentence=frame.format(w1),
                            w1=w1, w2=w2, tokens=tokens, p1=p1, p2=p2))

n_pairs = len(stimuli) // 2
print(f"Kept {n_pairs} matched pairs ({len(stimuli)} sentences).")
print("Dropped for tokenisation:", dropped or "none")

Kept 18 matched pairs (36 sentences).
Dropped for tokenisation: none


## Manipulation check: is word 2 really more predictable after the predictable word 1?

This verifies the stimuli against *this* model rather than my intuition. If a "predictable" pair isn't actually predicted by Gemma-2B, it shouldn't be counted as one.

In [ ]:
for s in stimuli:
    logp = model(s["tokens"]).log_softmax(-1)[0]
    toks = s["tokens"][0]
    s["logp_w2"] = logp[s["p1"], toks[s["p2"]]].item()          # log P(word 2 | ... word 1)
    s["logp_next"] = logp[s["p2"], toks[s["p2"] + 1]].item()    # log P(token after word 2 | ... word 2)

manip = (pd.DataFrame(stimuli)
         .pivot(index="pair_id", columns="condition", values="logp_w2")[["predictable", "control"]])
manip["pair"] = [f"{PAIRS[i][1]}/{PAIRS[i][2]} {PAIRS[i][3]}" for i in manip.index]
display(manip.assign(P_pred=np.exp(manip.predictable), P_ctrl=np.exp(manip.control)).round(3))

wins = int((manip.predictable > manip.control).sum())
print(f"Word 2 more likely after the predictable word 1 in {wins}/{n_pairs} pairs; "
      f"median P(word 2) = {np.exp(manip.predictable.median()):.3f} vs {np.exp(manip.control.median()):.3f}")

condition,predictable,control,pair,P_pred,P_ctrl
pair_id,,,,,
0,-0.118,-5.221,peanut/cold butter,0.889,0.005
1,-0.369,-5.228,ice/thick cream,0.692,0.005
2,-0.068,-1.741,credit/plastic card,0.935,0.175
3,-0.147,-4.731,traffic/long jam,0.863,0.009
4,-0.048,-3.800,swimming/large pool,0.953,0.022
5,-0.640,-3.573,birthday/small party,0.527,0.028
6,-0.181,-7.276,coffee/new shop,0.834,0.001
7,-0.067,-2.628,high/local school,0.935,0.072
8,-0.021,-2.625,living/quiet room,0.979,0.072


Word 2 more likely after the predictable word 1 in 17/18 pairs; median P(word 2) = 0.824 vs 0.025


## Ablation machinery

`feature_contribution` gets a feature's contribution to the residual stream as `decode(acts) − decode(acts without f)`, which stays correct whatever normalisation the SAE uses. The SAE's reconstruction error is left in place, so only the one feature's direction is removed.

In [ ]:
def encode_both(tokens):
    _, cache = model.run_with_cache(tokens, names_filter=[EARLY_HOOK, LATE_HOOK],
                                    stop_at_layer=LATE_LAYER + 1)
    return sae_early.encode(cache[EARLY_HOOK])[0], sae_late.encode(cache[LATE_HOOK])[0]

def top_features_per_position(acts, k):
    """(feature, position) for the top-k active features at each position >= 1. Position 0 is never used."""
    out = []
    for p in range(1, acts.shape[0]):
        vals, idxs = acts[p].topk(k)
        out += [(int(i), p) for v, i in zip(vals, idxs) if v > 0]
    return out

def feature_contribution(sae, acts_row, f):
    without = acts_row.clone()
    without[f] = 0
    return (sae.decode(acts_row[None]) - sae.decode(without[None]))[0]

def random_like(vec, gen):
    r = torch.randn(vec.shape, generator=gen).to(vec.device, vec.dtype)
    return r / r.norm() * vec.norm()

def ablation_hook(positions, vecs):
    """In copy b of the batch, subtract vecs[b] from the residual stream at positions[b] only."""
    rows = torch.arange(len(positions), device=vecs.device)
    pos = torch.tensor(positions, device=vecs.device)
    def hook(resid, hook):
        resid[rows, pos, :] = resid[rows, pos, :] - vecs
        return resid
    return hook

def late_features_after_ablation(tokens, positions, vecs):
    captured = {}
    def capture(resid, hook):
        captured["late"] = resid.clone()
    model.run_with_hooks(tokens.expand(len(positions), -1),
                         fwd_hooks=[(EARLY_HOOK, ablation_hook(positions, vecs)), (LATE_HOOK, capture)],
                         stop_at_layer=LATE_LAYER + 1)
    return sae_late.encode(captured["late"])          # [batch, seq, d_sae]

def logprobs_after_ablation(tokens, positions, vecs):
    logits = model.run_with_hooks(tokens.expand(len(positions), -1),
                                  fwd_hooks=[(EARLY_HOOK, ablation_hook(positions, vecs))])
    return logits.log_softmax(-1)                      # [batch, seq, vocab]

n_planned = sum(len(top_features_per_position(encode_both(s["tokens"])[0], K_PER_POS)) for s in stimuli)
print(f"{n_planned} feature ablations planned (+ the same number of random controls)")

1620 feature ablations planned (+ the same number of random controls)


## Run every ablation

In [ ]:
gen = torch.Generator().manual_seed(SEED)
T_MIN = min(THRESHOLDS)
edge_rows, word_link_rows = [], []
max_leak = 0.0

for s_idx, s in enumerate(tqdm(stimuli)):
    tokens, p1, p2 = s["tokens"], s["p1"], s["p2"]
    early, late = encode_both(tokens)
    cands = top_features_per_position(early, K_PER_POS)
    late_feats = sorted({g for g, _ in top_features_per_position(late, K_PER_POS)})
    base = late[:, late_feats]                                   # [seq, L]
    peak = base[1:].max(0).values.clamp_min(1e-6)                # each late feature's peak over positions >= 1

    real_vecs = torch.stack([feature_contribution(sae_early, early[p], f) for f, p in cands])
    rand_vecs = torch.stack([random_like(v, gen) for v in real_vecs])
    jobs = ([(f, p, "feature", v) for (f, p), v in zip(cands, real_vecs)]
            + [(f, p, "random", v) for (f, p), v in zip(cands, rand_vecs)])

    for start in range(0, len(jobs), BATCH):
        chunk = jobs[start:start + BATCH]
        patched = late_features_after_ablation(tokens, [j[1] for j in chunk],
                                               torch.stack([j[3] for j in chunk]))[:, :, late_feats]
        rel = ((base[None] - patched) / peak).cpu()               # share of peak lost, [b, seq, L]
        for b, (f, p, kind, _) in enumerate(chunk):
            r = rel[b]
            max_leak = max(max_leak, r[:p].abs().max().item())    # earlier positions must not change
            for dq, j in (r[p:].abs() >= T_MIN).nonzero().tolist():
                edge_rows.append(dict(s_idx=s_idx, pair_id=s["pair_id"], condition=s["condition"], kind=kind,
                                      f=f, p=p, g=late_feats[j], q=p + dq, dist=dq, rel=r[p + dq, j].item()))
            if p == p1:                                           # source is word 1
                j2 = int(r[p2].argmax())
                word_link_rows.append(dict(s_idx=s_idx, pair_id=s["pair_id"], condition=s["condition"], kind=kind,
                                           f=f, early_act=early[p1, f].item(), g=late_feats[j2],
                                           cross=r[p2, j2].item(), same=r[p1].max().item()))

edges = pd.DataFrame(edge_rows)
word_links = pd.DataFrame(word_link_rows)
print(f"{len(edges)} links with effect >= {T_MIN}")
print(f"Causality check - largest effect at a position BEFORE the ablation: {max_leak:.2e} (should be ~0)")

  0%|          | 0/36 [00:00<?, ?it/s]

## (A) Are cross-position links more than noise?

Each row counts strong links at one threshold. *Feature* rows ablate a real SAE feature; *random* rows ablate a random direction of the same length at the same place. The two have the same number of ablations, so their counts compare directly.

In [ ]:
rows = []
for t in THRESHOLDS:
    strong = edges[edges.rel.abs() >= t]
    for kind in ["feature", "random"]:
        k = strong[strong.kind == kind]
        rows.append(dict(threshold=t, ablation=kind, strong_links=len(k),
                         same_position=int((k.dist == 0).sum()), cross_position=int((k.dist > 0).sum())))
summary_A = pd.DataFrame(rows)
summary_A["cross_share"] = (summary_A.cross_position / summary_A.strong_links.clip(lower=1)).round(3)
display(summary_A)

main = edges[(edges.rel.abs() >= MAIN_T) & (edges.kind == "feature")]
cross = main[main.dist > 0]
print(f"\nCross-position links at threshold {MAIN_T}, by distance (1 = next token):")
display(cross.dist.clip(upper=4).value_counts().sort_index().rename(index={4: "4+"}).to_frame("links"))

In [ ]:
def describe(row):
    toks = model.to_str_tokens(stimuli[row.s_idx]["tokens"])
    return f"L{EARLY_LAYER}:{row.f} ({toks[row.p]!r}, pos {row.p}) -> L{LATE_LAYER}:{row.g} ({toks[row.q]!r}, pos {row.q})"

top_cross = cross.reindex(cross.rel.abs().sort_values(ascending=False).index).head(15)
print("Strongest cross-position links (real features):")
for row in top_cross.itertuples():
    print(f"  {describe(row):70s} effect={row.rel:+.2f}  [{stimuli[row.s_idx]['sentence']}]")

## (B) Is the word 1 → word 2 link stronger for predictable pairs?

For each sentence, the **link score** is the largest share of peak that any word-2 layer-12 feature loses when one of word 1's layer-6 features is ablated. A Wilcoxon signed-rank test compares matched pairs (same frame, same word 2).

In [ ]:
link = (word_links.groupby(["pair_id", "kind", "condition"]).cross.max()
        .unstack(["kind", "condition"]))
link.columns = [f"{k}_{c}" for k, c in link.columns]
link = link.dropna(subset=["feature_predictable", "feature_control"])
link["pair"] = [f"{PAIRS[i][1]}/{PAIRS[i][2]} {PAIRS[i][3]}" for i in link.index]
display(link.round(3))

def paired_test(a, b):
    d = (a - b).dropna()
    if len(d) < 6 or (d == 0).all():
        return np.nan
    return stats.wilcoxon(d).pvalue

p_B = paired_test(link.feature_predictable, link.feature_control)
p_B_rand = paired_test(link.random_predictable, link.random_control)
wins_B = int((link.feature_predictable > link.feature_control).sum())
print(f"Real features: predictable > control in {wins_B}/{len(link)} pairs; "
      f"median link {link.feature_predictable.median():.3f} vs {link.feature_control.median():.3f}; Wilcoxon p = {p_B:.3g}")
print(f"Random control: median {link.random_predictable.median():.3f} vs {link.random_control.median():.3f}; Wilcoxon p = {p_B_rand:.3g}")

## (C) Next-token check: does that word-1 feature matter for prediction?

For each sentence, take the word-1 feature with the strongest link to word 2, ablate it at word 1 only, and run the **full** model. Two numbers per sentence (negative = the model got worse):

- **Δ log P(word 2)**: the prediction made at word 1's position.
- **Δ log P(next)**: the prediction made at word 2's position, for the token after it. A change here can only come through a cross-position path, since the ablation was at word 1.

In [ ]:
gen_c = torch.Generator().manual_seed(SEED + 1)
rows = []
for s_idx, s in enumerate(stimuli):
    cand = word_links[(word_links.s_idx == s_idx) & (word_links.kind == "feature")]
    if cand.empty:
        continue
    f = int(cand.loc[cand.cross.idxmax(), "f"])
    early, _ = encode_both(s["tokens"])
    vec = feature_contribution(sae_early, early[s["p1"]], f)
    vecs = torch.stack([vec, random_like(vec, gen_c)])
    logp = logprobs_after_ablation(s["tokens"], [s["p1"], s["p1"]], vecs)
    toks = s["tokens"][0]
    for b, kind in enumerate(["feature", "random"]):
        rows.append(dict(pair_id=s["pair_id"], condition=s["condition"], kind=kind, f=f,
                         d_logp_w2=logp[b, s["p1"], toks[s["p2"]]].item() - s["logp_w2"],
                         d_logp_next=logp[b, s["p2"], toks[s["p2"] + 1]].item() - s["logp_next"]))
nextok = pd.DataFrame(rows)
display(nextok.groupby(["kind", "condition"])[["d_logp_w2", "d_logp_next"]].median().round(3))

wide = nextok[nextok.kind == "feature"].pivot(index="pair_id", columns="condition", values="d_logp_w2")
p_C = paired_test(wide.control, wide.predictable)   # predictable should drop more (more negative)
print(f"Δ log P(word 2), real features: Wilcoxon p (predictable vs control) = {p_C:.3g}")

## Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# (A) cross-position links: real features vs random directions, at each threshold
ax = axes[0]
x = np.arange(len(THRESHOLDS))
for i, kind in enumerate(["feature", "random"]):
    vals = summary_A[summary_A.ablation == kind].set_index("threshold").loc[THRESHOLDS, "cross_position"]
    bars = ax.bar(x + (i - 0.5) * 0.38, vals.values, width=0.36, label=f"{kind} ablation",
                  color=FEATURE_COLOR if kind == "feature" else RANDOM_COLOR)
    ax.bar_label(bars, padding=2, fontsize=9)
ax.set_xticks(x, [f"≥ {t:.0%}" for t in THRESHOLDS])
ax.set_xlabel("effect threshold (share of peak lost)")
ax.set_ylabel("cross-position links")
ax.set_title("(A) Cross-position links vs random control")
ax.legend(frameon=False)

def paired_plot(ax, left, right, title, ylabel, ref=None, log=False):
    for l, r in zip(left, right):
        ax.plot([0, 1], [l, r], color="#c3c2b7", lw=1, zorder=1)
    ax.scatter(np.zeros(len(left)), left, color=PRED_COLOR, s=36, zorder=2)
    ax.scatter(np.ones(len(right)), right, color=CTRL_COLOR, s=36, zorder=2)
    if ref is not None:
        ax.axhline(ref, color=RANDOM_COLOR, ls="--", lw=1, label="random-direction median")
        ax.legend(frameon=False, loc="best")
    ax.set_xticks([0, 1], ["predictable", "control"])
    ax.set_xlim(-0.4, 1.4)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale("log")

paired_plot(axes[1], np.exp(manip.predictable), np.exp(manip.control),
            "Manipulation check", "P(word 2 | word 1), log scale", log=True)
paired_plot(axes[2], link.feature_predictable, link.feature_control,
            "(B) Word 1 → word 2 link", "share of word-2 feature peak lost",
            ref=pd.concat([link.random_predictable, link.random_control]).median())

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", lw=0.8)
    ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("experiment2_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary (copy the numbers into the README)

In [ ]:
fa = summary_A[(summary_A.threshold == MAIN_T) & (summary_A.ablation == "feature")].iloc[0]
ra = summary_A[(summary_A.threshold == MAIN_T) & (summary_A.ablation == "random")].iloc[0]
print(f"""Gemma-2B, layer {EARLY_LAYER} -> {LATE_LAYER}, {n_pairs} matched pairs ({len(stimuli)} sentences), K_PER_POS={K_PER_POS}, threshold {MAIN_T}.
Manipulation check: word 2 more likely after the predictable word 1 in {wins}/{n_pairs} pairs.
(A) Real features: {fa.strong_links} strong links, {fa.same_position} same-position, {fa.cross_position} cross-position ({fa.cross_share:.1%}).
    Random directions: {ra.strong_links} strong links, {ra.cross_position} cross-position.
(B) Word 1 -> word 2 link: predictable > control in {wins_B}/{len(link)} pairs, Wilcoxon p = {p_B:.3g}.
(C) Delta log P(word 2) after ablating the linked word-1 feature: Wilcoxon p = {p_C:.3g}.
Causality check (should be ~0): {max_leak:.1e}""")

## Reading the results honestly

- **Same-position links dominating** is expected and not very interesting on its own: a direction removed at layer 6 partly survives to layer 12 at the same token. The comparison with random directions tells you whether real features do more than a random push of the same size.
- **Support for the hypothesis** needs (A) clearly more cross-position links for real features than random ones, *and* (B) a consistent predictable > control difference. One without the other is weak.
- **A null result is a result.** If (B) shows no difference, the Experiment 1 links were probably not about bigram predictability, and that is worth writing down.

## Limitations

- Candidate features are the top `K_PER_POS` at each position, so weaker features that might carry cross-position information are never tested.
- Control words differ from predictable ones in frequency and meaning, not just predictability.
- 18 pairs is enough to see a large, consistent effect, not a subtle one.
- Only one layer pair (6 → 12) is tested, and SAE reconstruction error is left in the residual stream.